# 基于Transformers的多项选择

## Step1 导入相关包

In [1]:
import evaluate
from datasets import DatasetDict
from transformers import AutoTokenizer, AutoModelForMultipleChoice, TrainingArguments, Trainer

## Step2 加载数据集

In [8]:
c3 = DatasetDict.load_from_disk("./c3/")
c3

DatasetDict({
    test: Dataset({
        features: ['id', 'context', 'question', 'choice', 'answer'],
        num_rows: 1625
    })
    train: Dataset({
        features: ['id', 'context', 'question', 'choice', 'answer'],
        num_rows: 11869
    })
    validation: Dataset({
        features: ['id', 'context', 'question', 'choice', 'answer'],
        num_rows: 3816
    })
})

In [17]:
c3["train"][:4]

{'id': [0, 1, 2, 3],
 'context': [['男：你今天晚上有时间吗?我们一起去看电影吧?', '女：你喜欢恐怖片和爱情片，但是我喜欢喜剧片，科幻片一般。所以……'],
  ['男：足球比赛是明天上午八点开始吧?', '女：因为天气不好，比赛改到后天下午三点了。'],
  ['女：今天下午的讨论会开得怎么样?', '男：我觉得发言的人太少了。'],
  ['男：我记得你以前很爱吃巧克力，最近怎么不吃了，是在减肥吗?', '女：是啊，我希望自己能瘦一点儿。']],
 'question': ['女的最喜欢哪种电影?',
  '根据对话，可以知道什么?',
  '关于这次讨论会，我们可以知道什么?',
  '女的为什么不吃巧克力了?'],
 'choice': [['恐怖片', '爱情片', '喜剧片', '科幻片'],
  ['今天天气不好', '比赛时间变了', '校长忘了时间'],
  ['会是昨天开的', '男的没有参加', '讨论得不热烈', '参加的人很少'],
  ['刷牙了', '要减肥', '口渴了', '吃饱了']],
 'answer': ['喜剧片', '比赛时间变了', '讨论得不热烈', '要减肥']}

In [25]:
len(c3["train"]['choice'])

11869

In [22]:
c3.pop("test")

Dataset({
    features: ['id', 'context', 'question', 'choice', 'answer'],
    num_rows: 1625
})

In [23]:
c3

DatasetDict({
    train: Dataset({
        features: ['id', 'context', 'question', 'choice', 'answer'],
        num_rows: 11869
    })
    validation: Dataset({
        features: ['id', 'context', 'question', 'choice', 'answer'],
        num_rows: 3816
    })
})

## Step3 数据集预处理

In [4]:
tokenizer = AutoTokenizer.from_pretrained("../../model/hfl/chinese-macbert-base")
tokenizer

BertTokenizerFast(name_or_path='../../model/hfl/chinese-macbert-base', vocab_size=21128, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=True),  added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}

In [18]:
def process_function(examples):
    # examples, dict, keys: ["context", "quesiton", "choice", "answer"]
    # examples, 1000
    context = []
    question_choice = []
    labels = []
    for idx in range(len(examples["context"])):
        ctx = "\n".join(examples["context"][idx])
        question = examples["question"][idx]
        choices = examples["choice"][idx]
        for choice in choices:
            context.append(ctx)
            question_choice.append(question + " " + choice)
        if len(choices) < 4:
            for _ in range(4 - len(choices)):
                context.append(ctx)
                question_choice.append(question + " " + "不知道")
        labels.append(choices.index(examples["answer"][idx]))
    tokenized_examples = tokenizer(context, question_choice, truncation="only_first", max_length=256, padding="max_length")     # input_ids: 4000 * 256, 
    tokenized_examples = {k: [v[i: i + 4] for i in range(0, len(v), 4)] for k, v in tokenized_examples.items()}     # 1000 * 4 *256
    tokenized_examples["labels"] = labels
    return tokenized_examples
        

In [24]:
res = c3["train"].select(range(10)).map(process_function, batched=True)
res

Dataset({
    features: ['id', 'context', 'question', 'choice', 'answer', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 10
})

In [25]:
import numpy as np
np.array(res["input_ids"]).shape

(10, 4, 256)

In [26]:
tokenized_c3 = c3.map(process_function, batched=True)
tokenized_c3

Map:   0%|          | 0/11869 [00:00<?, ? examples/s]

Map:   0%|          | 0/3816 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'context', 'question', 'choice', 'answer', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 11869
    })
    validation: Dataset({
        features: ['id', 'context', 'question', 'choice', 'answer', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 3816
    })
})

## Step4 创建模型

In [28]:
model = AutoModelForMultipleChoice.from_pretrained("../../model/hfl/chinese-macbert-base")

Some weights of BertForMultipleChoice were not initialized from the model checkpoint at ../../model/hfl/chinese-macbert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## Step5 创建评估函数

In [30]:
import numpy as np
accuracy = evaluate.load("../../evaluate/metrics/accuracy")

def compute_metric(pred):
    predictions, labels = pred
    predictions = np.argmax(predictions, axis=-1)
    return accuracy.compute(predictions=predictions, references=labels)

## Step6 配置训练参数

In [34]:
args = TrainingArguments(
    output_dir="./muliple_choice",
    per_device_train_batch_size=8,  # 8 * 2 * 4 = 64
    gradient_accumulation_steps=2,
    gradient_checkpointing=False,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=True
)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


## Step7 创建训练器

In [36]:
trainer = Trainer(
    model=model,
    args=args,
    tokenizer=tokenizer,
    train_dataset=tokenized_c3["train"],
    eval_dataset=tokenized_c3["validation"],
    compute_metrics=compute_metric
)

## Step8 模型训练

In [37]:
trainer.train()

  0%|          | 0/742 [00:00<?, ?it/s]

{'loss': 1.3105, 'grad_norm': 8.864510536193848, 'learning_rate': 4.669811320754717e-05, 'epoch': 0.07}
{'loss': 1.1642, 'grad_norm': 10.124369621276855, 'learning_rate': 4.33288409703504e-05, 'epoch': 0.13}
{'loss': 1.1124, 'grad_norm': 11.313867568969727, 'learning_rate': 3.995956873315364e-05, 'epoch': 0.2}
{'loss': 1.1027, 'grad_norm': 11.604186058044434, 'learning_rate': 3.6590296495956876e-05, 'epoch': 0.27}
{'loss': 1.0739, 'grad_norm': 19.100595474243164, 'learning_rate': 3.322102425876011e-05, 'epoch': 0.34}
{'loss': 1.0815, 'grad_norm': 8.205470085144043, 'learning_rate': 2.9851752021563344e-05, 'epoch': 0.4}
{'loss': 1.0353, 'grad_norm': 8.673521041870117, 'learning_rate': 2.6482479784366575e-05, 'epoch': 0.47}
{'loss': 0.9692, 'grad_norm': 11.479671478271484, 'learning_rate': 2.3113207547169813e-05, 'epoch': 0.54}
{'loss': 0.9875, 'grad_norm': 16.506895065307617, 'learning_rate': 1.9743935309973047e-05, 'epoch': 0.61}
{'loss': 0.9949, 'grad_norm': 12.532753944396973, 'learn

  0%|          | 0/477 [00:00<?, ?it/s]

{'eval_loss': 0.910537600517273, 'eval_accuracy': 0.6027253668763103, 'eval_runtime': 83.487, 'eval_samples_per_second': 45.708, 'eval_steps_per_second': 5.713, 'epoch': 1.0}
{'train_runtime': 817.9854, 'train_samples_per_second': 14.51, 'train_steps_per_second': 0.907, 'train_loss': 1.041149447870383, 'epoch': 1.0}


TrainOutput(global_step=742, training_loss=1.041149447870383, metrics={'train_runtime': 817.9854, 'train_samples_per_second': 14.51, 'train_steps_per_second': 0.907, 'total_flos': 6245674154244096.0, 'train_loss': 1.041149447870383, 'epoch': 1.0})

## Step9 模型预测

In [ ]:
from typing import Any
import torch


class MultipleChoicePipeline:

    def __init__(self, model, tokenizer) -> None:
        self.model = model
        self.tokenizer = tokenizer
        self.device = model.device

    def preprocess(self, context, quesiton, choices):
        cs, qcs = [], []
        for choice in choices:
            cs.append(context)
            qcs.append(quesiton + " " + choice)
        return tokenizer(cs, qcs, truncation="only_first", max_length=256, return_tensors="pt", padding=True)   # 不padding的话:答案出现不同长度的情况会报错（pipeline）

    def predict(self, inputs):
        inputs = {k: v.unsqueeze(0).to(self.device) for k, v in inputs.items()}
        return self.model(**inputs).logits

    def postprocess(self, logits, choices):
        predition = torch.argmax(logits, dim=-1).cpu().item()
        return choices[predition]

    def __call__(self, context, question, choices) -> Any:
        inputs = self.preprocess(context, question, choices)
        logits = self.predict(inputs)
        result = self.postprocess(logits, choices)
        return result

In [44]:
pipe = MultipleChoicePipeline(model, tokenizer)

In [45]:
pipe("小明在北京上班", "小明在哪里上班？", ["北京", "上海", "河北", "海南", "河北的", "海南"])

'北京'